# E045 — Score-Level Ensemble: MFCC + LPCC + PLP

**Motivation:** E022 tried MFCC+LPCC fusion but collapsed to LPCC alone. E021 (PLP) was rejected at 5.56%, but PLP may add complementary signal when combined with MFCC+LPCC.

**Hypothesis:** Ensemble of 3 feature types (MFCC, LPCC, PLP) with calibrated score averaging will outperform any single feature type.

**Approach:**
1. Train 3 separate UBM-MAP systems (MFCC, LPCC, PLP)
2. Calibrate each stream (Platt scaling)
3. Average calibrated scores
4. Compare to best single system (E042: 0.46%)

**Configs:**
- `LPCC (E042)`: Baseline (tied cov + speedTTA)
- `MFCC+LPCC`: 2-system ensemble
- `MFCC+LPCC+PLP`: 3-system ensemble

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
import librosa
from scipy.special import logsumexp
from sklearn.mixture import GaussianMixture
from sklearn.linear_model import LogisticRegression
import copy

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

SEED = 67
UBM_COMPONENTS = 32
MAP_R = 16.0

DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
y_all = manifest['label'].to_numpy()
print(f'{len(manifest)} samples')

E042_REF = {'mean_eer': 0.46, 'std_eer': 0.65}

In [ ]:
def _find_wav(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".wav")
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _extract_mfcc(y, sr):
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    feat = np.vstack([mfcc, delta, delta2]).T
    feat -= feat.mean(axis=0)
    return feat

def _extract_lpcc(y, sr, order=12, n_cep=13, hop_length=160, win_length=400):
    frames = librosa.util.frame(y, frame_length=win_length, hop_length=hop_length)
    cep_frames = []
    for frame in frames.T:
        frame = frame * np.hanning(len(frame))
        try:
            a = librosa.lpc(frame.astype(np.float64), order=order)
            A_freq = np.fft.rfft(a, n=512)
            log_H = -np.log(np.abs(A_freq) + 1e-10)
            cep = np.real(np.fft.irfft(log_H))[:n_cep]
        except Exception:
            cep = np.zeros(n_cep)
        cep_frames.append(cep)
    feat = np.array(cep_frames, dtype=np.float32)
    delta = librosa.feature.delta(feat.T).T
    delta2 = librosa.feature.delta(feat.T, order=2).T
    feat = np.hstack([feat, delta, delta2])
    feat -= feat.mean(axis=0)
    return feat

def _extract_plp(y, sr, order=12, n_bands=20):
    # Simplified PLP: Bark filterbank + equal loudness + cube root + LPC
    n_fft = 512
    hop = 160
    frames = librosa.util.frame(y, frame_length=400, hop_length=hop)
    cep_frames = []
    # Bark bands (simplified)
    bark_edges = [0, 100, 200, 300, 400, 510, 640, 790, 960, 1150, 1370, 1620, 1900, 2220, 2580, 2990, 3450, 3970, 4570, 5260, 6400]
    for frame in frames.T:
        frame = frame * np.hanning(len(frame))
        spec = np.abs(np.fft.rfft(frame, n=n_fft))
        # Bark integration (simplified)
        bark_energies = []
        for i in range(len(bark_edges)-1):
            idx_start = int(bark_edges[i] / (sr/2) * (n_fft/2))
            idx_end = int(bark_edges[i+1] / (sr/2) * (n_fft/2))
            bark_energies.append(np.sum(spec[idx_start:idx_end]**2) + 1e-10)
        bark_energies = np.array(bark_energies)
        # Equal loudness (simplified - pre-emphasis)
        bark_energies *= np.linspace(1, 0.5, n_bands)
        # Cube root compression
        bark_energies = bark_energies ** (1/3)
        # LPC on Bark spectrum
        try:
            a = librosa.lpc(bark_energies, order)
            cep = np.real(np.fft.irfft(np.exp(-np.log(np.abs(np.fft.rfft(a, 64)) + 1e-10))))[:13]
        except:
            cep = np.zeros(13)
        cep_frames.append(cep)
    feat = np.array(cep_frames, dtype=np.float32)
    delta = librosa.feature.delta(feat.T).T
    delta2 = librosa.feature.delta(feat.T, order=2).T
    feat = np.hstack([feat, delta, delta2])
    feat -= feat.mean(axis=0)
    return feat

def _train_ubm(X, cov_type='tied'):
    return GaussianMixture(n_components=UBM_COMPONENTS, covariance_type=cov_type, max_iter=200, random_state=SEED).fit(X)

def _map_adapt(ubm, X_target):
    log_resp = ubm._estimate_log_prob(X_target) + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp = np.exp(log_resp)
    n_k = resp.sum(axis=0)
    mu_hat = (resp.T @ X_target) / (n_k[:, None] + 1e-10)
    alpha = n_k / (n_k + MAP_R)
    adapted = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    return adapted

print('Feature functions defined')

## 2. Cross-validation

In [ ]:
print("Collecting OOF scores...")
oof_mfcc = np.full(len(manifest), np.nan)
oof_lpcc = np.full(len(manifest), np.nan)
oof_plp = np.full(len(manifest), np.nan)

for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    seed_f = SEED + fold_id
    train_df = manifest.loc[train_idx]
    val_df = manifest.loc[val_idx]
    print(f"Fold {fold_id}...")
    
    rng = np.random.default_rng(seed_f)
    
    # Extract features for all types
    for feat_name, extract_fn in [('mfcc', _extract_mfcc), ('lpcc', _extract_lpcc), ('plp', _extract_plp)]:
        X_tgt, X_non = [], []
        for _, row in train_df.iterrows():
            y_wav, sr = librosa.load(_find_wav(row["stem"], DATA), sr=None, mono=True)
            wavs = [y_wav]
            if row["label"] == 1 and feat_name != 'plp':  # PLP: no pitch aug (E021 failed)
                wavs.append(librosa.effects.time_stretch(y_wav, rate=rng.uniform(0.9, 1.1)))
            for y_aug in wavs:
                feat = extract_fn(y_aug, sr)
                if row["label"] == 1:
                    X_tgt.append(feat)
                else:
                    X_non.append(feat)
        
        ubm = _train_ubm(np.vstack(X_non))
        adapted = _map_adapt(ubm, np.vstack(X_tgt))
        
        for _, row in val_df.iterrows():
            y_wav, sr = librosa.load(_find_wav(row["stem"], DATA), sr=None, mono=True)
            feat = extract_fn(y_wav, sr)
            score = adapted.score(feat) - ubm.score(feat)
            if feat_name == 'mfcc':
                oof_mfcc[row.name] = score
            elif feat_name == 'lpcc':
                oof_lpcc[row.name] = score
            else:
                oof_plp[row.name] = score

# Calibrate
def calibrate(scores, labels):
    cal = LogisticRegression(C=1e6, max_iter=1000, class_weight='balanced')
    cal.fit(scores.reshape(-1, 1), labels)
    return cal

cal_mfcc = calibrate(oof_mfcc, y_all)
cal_lpcc = calibrate(oof_lpcc, y_all)
cal_plp = calibrate(oof_plp, y_all)

cal_m = cal_mfcc.decision_function(oof_mfcc.reshape(-1, 1))
cal_l = cal_lpcc.decision_function(oof_lpcc.reshape(-1, 1))
cal_p = cal_plp.decision_function(oof_plp.reshape(-1, 1))

# Evaluate
ensembles = {
    'LPCC (E042)': cal_l,
    'MFCC+LPCC': 0.5 * cal_m + 0.5 * cal_l,
    'MFCC+LPCC+PLP': (cal_m + cal_l + cal_p) / 3,
}

print("\n=== Ensemble Results ===")
for name, scores in ensembles.items():
    eer, _ = compute_eer(scores[y_all == 1], scores[y_all == 0])
    print(f"{name:20s}: EER={eer*100:5.2f}%")

## 3. Conclusion

Ensemble effect: [TBD]
Decision: [Adopt/Reject]